In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import os
import joblib
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

In [ ]:
NUMERIC_FEATURES = [
    "speed",
    "current_engine_rpm",
    "acceleration_magnitude",
    "velocity_magnitude",
    "tire_stress_front",
    "tire_stress_rear",
    "wheel_slip_magnitude_front",
    "wheel_slip_magnitude_rear",
    "avg_tire_temp",
    "power",
    "torque",
    "boost",
    "yaw",
    "pitch",
    "roll",
    "steer",
    "rpm_speed_ratio"
]

CATEGORICAL_FEATURES = [
    "gear",
    "lap_number",
    "race_position"
]

ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

In [ ]:
train = pd.read_csv("data/splits/train.csv")
val   = pd.read_csv("data/splits/val.csv")

In [ ]:
TARGET = "gear"
FEATURES_NO_GEAR = [f for f in ALL_FEATURES if f != "gear"]

X_train = train[FEATURES_NO_GEAR]
y_train = train[TARGET]

X_val = val[FEATURES_NO_GEAR]
y_val = val[TARGET]

In [ ]:
print("Featire no gear:", FEATURES_NO_GEAR)

In [ ]:
model = RandomForestClassifier(
    n_estimators=20,
    max_depth=4,
    min_samples_leaf=100,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)


In [ ]:
# Predictions
y_train_pred = model.predict(X_train)
y_val_pred   = model.predict(X_val)

print("TRAIN Accuracy:", accuracy_score(y_train, y_train_pred))
print("VAL   Accuracy:", accuracy_score(y_val, y_val_pred))

In [ ]:
print("\nValidation Classification Report:")
print(classification_report(y_val,y_val_pred))

In [ ]:
labels = sorted(y_val.unique())  # [1, 2, 3, 4]

plt.figure(figsize=(7, 6))
sns.heatmap(
    confusion_matrix(y_val, y_val_pred, labels=labels),
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels
)
plt.xlabel("Predicted Gear")
plt.ylabel("Actual Gear")
plt.title("Gear Prediction RandomForest")
plt.show()

In [ ]:
importances = pd.Series(
    model.feature_importances_,
    index=FEATURES_NO_GEAR
).sort_values(ascending=False)

print(importances.head(10))

importances.head(15).plot(
    kind="barh",
    figsize=(8,6),
    title="Top Feature Importances Gear Classification"
)
plt.gca().invert_yaxis()
plt.show()


In [ ]:
os.makedirs("artifacts/gear_classifier", exist_ok=True)
joblib.dump(model, "artifacts/gear_classifier/rf_model.pkl")
print("Gear classifier saved.")

In [ ]:
import joblib

# Load model
loaded_model = joblib.load("artifacts/gear_classifier/rf_model.pkl")

# Test inference
sample = X_val.iloc[:5]
preds = loaded_model.predict(sample)

print("Preds:", preds)
print("Actual:", y_val.iloc[:5].values)